In [1]:
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import re
import pickle
from collections import Counter, defaultdict
from tqdm import tqdm
import math

SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        
        # Splits on whitespace+word or just whitespace. 
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        # 1. Setup base vocabulary
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i

        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1

        # 2. Split text into chunks and get frequencies
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        
        # 3. Store each distinct chunk as a tuple of base byte ids
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }

        # 4. Initialize weighted pair counts and the pair-to-chunk index
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]

        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)

        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")

        # 5. Iteratively merge the best pair
        while next_id < target:
            if not pair_counts:
                break

            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break

            # Min tuple tie-breaker matching your original behavior
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)

            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id

            # 6. Update only the chunks containing the merged pair
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                
                # Subtract old pair combinations
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                
                # Apply the merge inside this chunk
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                
                # Add new pair combinations created by the merge
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)

            next_id += 1
            pbar.update(1)

        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        
        for chunk in chunks:
            # Check the dictionary cache first
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            
            # Cache miss: break into bytes and process
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            
            # Merge until no adjacent pair has a valid merge rank
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                
                # Find the pair with the lowest rank
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                
                if best_pair is None:
                    break  # No more valid merges for this chunk
                    
                # Apply the best merge to all occurrences in the chunk
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            
            # Store result in cache and append
            self.cache[chunk] = ids
            out_ids.extend(ids)
            
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    """Train the A1 BPE tokenizer on `train_text` and return it, ready for A3."""
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

NUM_SPECIAL = len(SPECIAL_TOKENS)

@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads   # how many query heads share each kv head
        self.d_head = d_model // n_heads
        # Preserve the Q1.4 hook in later questions. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: q_proj maps d_model -> n_heads * d_head (as before).
        # TODO: k_proj, v_proj map d_model -> n_kv_heads * d_head (SMALLER than before
        #       whenever n_kv_heads < n_heads -- this is where the parameter savings
        #       and (in a real serving system) the KV-cache savings come from).
        self.q_proj = nn.Linear(d_model,n_heads * self.d_head)
        self.k_proj = nn.Linear(d_model,n_kv_heads * self.d_head)
        self.v_proj = nn.Linear(d_model,n_kv_heads * self.d_head)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        # TODO 1: project q -> (B, n_heads, T, d_head); k, v -> (B, n_kv_heads, T, d_head)
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1,2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1,2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1,2)
        # TODO 2: expand k and v from n_kv_heads to n_heads by repeating each kv head
        #         `self.n_rep` times (torch.repeat_interleave along the head dim is the
        #         natural tool here) so shapes line up for a standard batched matmul.
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        # TODO 3: same attention as Q1, using (q @ k^T) * self.attention_scale,
        #         then causal mask + softmax + weighted sum.
        att = (q @ k.transpose(-2, -1)) * self.attention_scale
        att = att.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        # TODO 4: reshape back to (B, T, C), apply out_proj.
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        out = self.resid_dropout(self.out_proj(y))
        return out


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x

class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


In [2]:
# Cell A

import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-5-simple-deepseek/cs781_phase1_full"
device = torch.device("cuda")

tok = pickle.load(open(f"{ROOT}/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/legal-gpt2-final/ckpt.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); lm.load_state_dict(ck["model"]); lm.to(device)
del ck                                    # frees the 1.2 GB (optimizer state is not needed)
print("step", "params", sum(p.numel() for p in lm.parameters())/1e6, "M", "max_seq_len", cfg.max_seq_len)

step params 98.408448 M max_seq_len 2048


In [3]:
# Cell B

def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

In [4]:
# Cell C

import json
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

# Get your Hugging Face token from Kaggle Secrets
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

def get(name):
    p = hf_hub_download(
        repo_id="Exploration-Lab/CS-781-Capstone",
        filename=f"Phase_2_LNER/{name}",
        repo_type="dataset",
        token=hf_token
    )

    if p.endswith(".json"):
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    else:
        with open(p, "r", encoding="utf-8") as f:
            return f.read().split()

f1_text, f1_gold = get("Train/fold1-text.json"), get("Train/fold1-gold.json")
f2_text, f2_gold = get("Train/fold2-text.json"), get("Train/fold2-gold.json")
test_text = get("Test/fold3-text.json")
LABELS = get("Train/labels.txt")

print(len(f1_text), len(f2_text), len(test_text), LABELS)

k = next(iter(f1_gold))
print(k, f1_gold[k][:3])  # Check annotation format


fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

35 35 35 ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
115651329 [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011'], [276, 308, 'COURT', 'High Court of Punjab and Haryana']]


In [5]:
# Cell D

SPLIT = re.compile(r'\s*\S+|\s+')

def encode_with_offsets(text):
    ids, offs = [], []
    for m in SPLIT.finditer(text):
        chunk, cs = m.group(), m.start()
        b2c = []
        for ci, ch in enumerate(chunk):
            b2c += [ci] * len(ch.encode("utf-8"))
        b = 0
        for t in tok.encode(chunk):
            n = len(tok.id_to_bytes[t])
            s, e = cs + b2c[b], cs + b2c[b + n - 1] + 1
            ids.append(t); offs.append((s, e)); b += n
    return ids, offs

def content_start(text, s, e):            # skip the leading space a token carries
    while s < e and text[s].isspace(): s += 1
    return s

def spans(gold_doc):                      # -> list of (start, end, label)
    return [(int(g[0]), int(g[1]), g[2]) for g in gold_doc]

# Check 1: token ids match the Phase 1 tokenizer exactly
t = f1_text[k]; ids, offs = encode_with_offsets(t)
assert ids == tok.encode(t), "chunked encoding differs from tok.encode"

# Check 2: gold -> token BIO -> back to spans
def to_bio(text, offs, ents):
    tags = ["O"] * len(offs)
    for s, e, lab in ents:
        first = True
        for i, (ts, te) in enumerate(offs):
            cs_ = content_start(text, ts, te)
            if cs_ < e and te > s and cs_ < te:      # token overlaps the entity
                tags[i] = ("B-" if first else "I-") + lab; first = False
    return tags

def from_bio(text, offs, tags):
    out, cur = [], None
    for (ts, te), tg in zip(offs, tags):
        if tg.startswith("B-") or (tg.startswith("I-") and (cur is None or cur[2] != tg[2:])):
            if cur: out.append(tuple(cur))
            cur = [content_start(text, ts, te), te, tg[2:]]
        elif tg.startswith("I-"):
            cur[1] = te
        else:
            if cur: out.append(tuple(cur)); cur = None
    if cur: out.append(tuple(cur))
    return out

hit = tot = 0; bad = []
for texts, golds in [(f1_text, f1_gold), (f2_text, f2_gold)]:
    for d in golds:
        text = texts[d]; _, offs = encode_with_offsets(text)
        g = set(spans(golds[d])); r = set(from_bio(text, offs, to_bio(text, offs, g)))
        hit += len(g & r); tot += len(g)
        bad += [(text[s:e], lab, d) for s, e, lab in g - r][:3]
print(f"round-trip exact: {hit}/{tot} = {hit/tot:.2%}"); print(bad[:15])

round-trip exact: 6151/8393 = 73.29%
[('Mange Ram', 'WIT', '115651329'), ('High Court', 'COURT', '115651329'), ('IPC', 'STAT', '115651329'), ('06.08.2008', 'DATE', '37849282'), ('15.11.2003', 'DATE', '37849282'), ('19.07.2005', 'DATE', '37849282'), ('Civil Court', 'COURT', '975074'), ('Civil Court', 'COURT', '975074'), ('Civil Court', 'COURT', '975074'), ('9.8.1993', 'DATE', '189525449'), ('Surendra', 'APP', '189525449'), ('Evidence Act', 'STAT', '189525449'), ('High Court', 'COURT', '736324'), ('O.A. No. 202 of 1999', 'CASENO', '736324'), ('High Court', 'COURT', '736324')]


In [6]:
# Cell E

# =====================================================================================
# CELL E — LORMA fine-tuning of c-5 for Legal NER (template)
# Needs: Cell 0 (classes), A (lm, tok, cfg, device), B (features), C (data), D (offsets/BIO)
#
# What is left as TODO (your own work, per course rules):
#   E2  the LORMA layer itself (parameters + forward)
#   E3  the NER head
#   E1/E5 hyperparameters (window, rank, lr, epochs, ...)
# Everything else is plumbing: windows, batching, training loop, scoring, prediction.
# =====================================================================================
import math, time, json, zipfile, random
import torch, torch.nn as nn, torch.nn.functional as F

# -------------------------------------------------------------------------------------
# E1. Hyperparameters — YOU choose these (None = not set yet; the asserts stop you)
# -------------------------------------------------------------------------------------
WINDOW      = 200    # tokens per window (<= cfg.max_seq_len = 2048)
STRIDE      = 10    # new tokens per window (< WINDOW); WINDOW-STRIDE = left context overlap
LORMA_RANK  = 20      # Rank of the low-rank adaptation matrices
LR          = 3e-4   # Learning rate suitable for parameter-efficient fine-tuning
EPOCHS      = 10      # Usually 3-5 epochs are sufficient for NER with PEFT
BATCH_SIZE  = 64      # windows per step (2048-token windows on a T4: start small)
WEIGHT_DECAY = 0.0  # Standard slight weight decay
SEED        = 42
for name in ["WINDOW", "STRIDE", "LORMA_RANK", "LR", "EPOCHS", "BATCH_SIZE"]:
    assert globals()[name] is not None, f"set {name}"
assert STRIDE < WINDOW <= cfg.max_seq_len
random.seed(SEED); torch.manual_seed(SEED)

# Tag vocabulary: O + B-/I- for each of the 12 labels -> 25 tags
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
TAG2ID = {t: i for i, t in enumerate(TAGS)}
IGNORE = -100   # positions that are context only / padding: no loss


# -------------------------------------------------------------------------------------
# E1b. Documents -> overlapping windows
#   c-5 is causal (each token sees only its left), so every window after the first
#   carries WINDOW-STRIDE tokens of left context; only its last STRIDE tokens get
#   labels / predictions. Every token is therefore labelled exactly once.
# -------------------------------------------------------------------------------------
def make_windows(ids):
    """Return list of (start, end, first_labelled_index_inside_window)."""
    n, out, s = len(ids), [], 0
    while True:
        e = min(s + WINDOW, n)
        first = 0 if s == 0 else WINDOW - STRIDE
        out.append((s, e, first))
        if e >= n:
            return out
        s += STRIDE

def build_examples(texts, golds=None):
    """One example per window: token ids, tag ids (or None), and bookkeeping for decoding."""
    examples, doc_cache = [], {}
    for doc_id, text in texts.items():
        ids, offs = encode_with_offsets(text)
        tag_ids = None
        if golds is not None:
            tags = to_bio(text, offs, spans(golds.get(doc_id, [])))
            tag_ids = [TAG2ID[t] for t in tags]
        doc_cache[doc_id] = (text, offs)
        for (s, e, first) in make_windows(ids):
            lab = None
            if tag_ids is not None:
                lab = [IGNORE] * first + tag_ids[s + first:e]
            examples.append({"doc": doc_id, "start": s, "first": first,
                             "ids": ids[s:e], "labels": lab})
    return examples, doc_cache

def collate(batch):
    """Right-pad to the longest window in the batch (safe: causal model, pad is on the right)."""
    T = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    lab = torch.full((len(batch), T), IGNORE, dtype=torch.long)
    for i, b in enumerate(batch):
        ids[i, :len(b["ids"])] = torch.tensor(b["ids"])
        if b["labels"] is not None:
            lab[i, :len(b["labels"])] = torch.tensor(b["labels"])
    return ids, lab, batch

PAD_ID = 0   # "<pad>" in your tokenizer


# -------------------------------------------------------------------------------------
# E2. LORMA layer — core algorithm
#   Wraps one frozen nn.Linear of the backbone and adds a small trainable,
#   low-rank update to it. 
# -------------------------------------------------------------------------------------
class LORMALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base                         # frozen pretrained layer (W, b)
        for p in self.base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.rank = rank
        
        # LORMA parameters: A (rank x in_features) and B (out_features x rank)
        self.lorma_A = nn.Parameter(torch.empty(self.rank, self.in_features))
        self.lorma_B = nn.Parameter(torch.empty(self.out_features, self.rank))
        
        # Initialisation: A gets Kaiming uniform (to provide variance), B is initialized to zero
        # so that at step 0, B * A = 0 and the layer behaves exactly like the base layer.
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        # base(x) + (x @ A^T @ B^T)
        base_out = self.base(x)
        # Compute low-rank update efficiently: first project to rank, then to out_features
        lorma_out = F.linear(F.linear(x, self.lorma_A), self.lorma_B)
        return base_out + lorma_out


def apply_lorma(model, rank, target_names=("q_proj", "k_proj", "v_proj", "out_proj", "c_proj", "c_attn")):
    """Replace chosen linear layers inside every transformer block with LORMALinear.
    Expanded target_names to cover various naming conventions, and relaxed the strict nn.Linear check."""
    replaced = 0
    for blk in model.blocks:
        for name in target_names:
            parent = blk.attn
            if hasattr(parent, name):
                base = getattr(parent, name)
                # Instead of strict isinstance(base, nn.Linear), check for standard Linear attributes
                if hasattr(base, 'in_features') and hasattr(base, 'out_features'):
                    setattr(parent, name, LORMALinear(base, rank))
                    replaced += 1
                    
        # Optional: uncomment to also adapt FFN layers
        # if hasattr(blk, "ffn"):
        #     for name, base in blk.ffn.named_children():
        #         if hasattr(base, 'in_features') and hasattr(base, 'out_features'):
        #             setattr(blk.ffn, name, LORMALinear(base, rank))
        #             replaced += 1
                    
    return replaced


# -------------------------------------------------------------------------------------
# E3. NER model = c-5 backbone (frozen + LORMA) + token-classification head
# -------------------------------------------------------------------------------------
class NERModel(nn.Module):
    def __init__(self, backbone, n_tags):
        super().__init__()
        self.backbone = backbone
        
        # Head mapping 1024-dim token vectors to n_tags logits
        self.head = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(1024, n_tags)
        )

    def forward(self, idx):
        x = self.backbone.tok_emb(idx)                       # same path as Cell B features()
        if self.backbone.cfg.pos_emb_type == "sinusoidal":
            x = x + self.backbone.pos_emb[:idx.size(1), :]
        for blk in self.backbone.blocks:
            x = blk(x)
        h = self.backbone.ln_f(x)                            # (B, T, 1024) — LM head skipped
        return self.head(h)                                  # (B, T, n_tags)


# Reference features BEFORE adding LORMA (used for the identity-at-init check below)
lm.eval()
probe = torch.tensor([encode_with_offsets(next(iter(f1_text.values())))[0][:64]], device=device)
with torch.no_grad():
    ref_feats = features(probe).float()

# Freeze the whole backbone, then add LORMA (its params are the only trainable backbone params)
for p in lm.parameters():
    p.requires_grad = False
n_wrapped = apply_lorma(lm, LORMA_RANK)
model = NERModel(lm, len(TAGS)).to(device)
assert model.head is not None, "define the head in NERModel.__init__"

# Sanity check: at initialisation LORMA must leave the backbone unchanged
with torch.no_grad():
    diff = (features(probe).float() - ref_feats).abs().max().item()
print(f"LORMA identity-at-init check: max |diff| = {diff:.2e} (should be ~0)")

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"wrapped {n_wrapped} linears | trainable {n_train/1e6:.3f}M / total {n_total/1e6:.2f}M "
      f"({100*n_train/n_total:.2f}%)")


# -------------------------------------------------------------------------------------
# E4. Data: train on fold1, validate on fold2 (swap to fold1+fold2 for the final run)
# -------------------------------------------------------------------------------------
train_ex, _ = build_examples(f1_text, f1_gold)
val_ex, val_cache = build_examples(f2_text, f2_gold)
print(f"train windows {len(train_ex)} | val windows {len(val_ex)}")


# -------------------------------------------------------------------------------------
# E5. Loss — plain cross-entropy over tags, ignoring context/padding positions.
# -------------------------------------------------------------------------------------
CLASS_WEIGHTS = None   # Leaving None to use unweighted cross entropy initially

def loss_fn(logits, labels):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                           weight=CLASS_WEIGHTS, ignore_index=IGNORE)


# -------------------------------------------------------------------------------------
# E6. Prediction + strict scoring (plumbing)
# -------------------------------------------------------------------------------------
@torch.no_grad()
def predict(examples, cache):
    """Returns {doc_id: [(start_char, end_char, label), ...]}"""
    model.eval()
    doc_tags = {d: ["O"] * len(cache[d][1]) for d in cache}
    for i in range(0, len(examples), BATCH_SIZE):
        ids, _, batch = collate(examples[i:i + BATCH_SIZE])
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(ids.to(device)).argmax(-1).cpu()
        for b, ex in enumerate(batch):
            for j in range(ex["first"], len(ex["ids"])):
                doc_tags[ex["doc"]][ex["start"] + j] = TAGS[pred[b, j]]
    return {d: from_bio(cache[d][0], cache[d][1], doc_tags[d]) for d in cache}

def strict_macro_f1(pred, gold):
    """Exact (start, end, label) match; per-label P/R/F1, macro over the 12 labels."""
    tp = {l: 0 for l in LABELS}; fp = dict(tp); fn = dict(tp)
    for d in gold:
        g, p = set(spans(gold[d])), set(pred.get(d, []))
        for s, e, l in p & g: tp[l] += 1
        for s, e, l in p - g: fp[l] = fp.get(l, 0) + 1
        for s, e, l in g - p: fn[l] += 1
    used = [l for l in LABELS if tp[l] + fp[l] + fn[l] > 0]
    per = {}
    for l in used:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = (P, R, 2 * P * R / (P + R) if P + R else 0.0)
    return (sum(v[2] for v in per.values()) / len(per) if per else 0.0), per


# -------------------------------------------------------------------------------------
# E7. Training loop (fp16 autocast, AdamW on trainable params only)
# -------------------------------------------------------------------------------------
from tqdm.auto import tqdm

params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

scaler = torch.amp.GradScaler("cuda")
torch.cuda.reset_peak_memory_stats()
t0, best_f1, log = time.time(), -1.0, []

for epoch in range(EPOCHS):
    model.train(); random.shuffle(train_ex); tot = 0.0
    
    # Wrap the batch iterator with tqdm
    pbar = tqdm(range(0, len(train_ex), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for i in pbar:
        ids, lab, _ = collate(train_ex[i:i + BATCH_SIZE])
        ids, lab = ids.to(device), lab.to(device)
        
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(ids)
        loss = loss_fn(logits.float(), lab)
        
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update()
        
        loss_val = loss.item()
        tot += loss_val
        
        # Update progress bar suffix with current batch loss
        pbar.set_postfix(loss=f"{loss_val:.4f}")
        
    val_pred = predict(val_ex, val_cache)
    f1, per = strict_macro_f1(val_pred, f2_gold)
    log.append({"epoch": epoch, "train_loss": tot / max(1, math.ceil(len(train_ex) / BATCH_SIZE)),
                "val_macro_f1": f1})
    print(log[-1])
    
    if f1 > best_f1:
        best_f1 = f1
        torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
                   "/kaggle/working/c5_lorma_best_trainable.pt")   # only LORMA + head weights

print(f"best val strict macro-F1 {best_f1:.4f} | time {(time.time()-t0)/60:.1f} min | "
      f"peak GPU mem {torch.cuda.max_memory_allocated()/2**30:.2f} GB")
for l, (P, R, F1) in per.items():
    print(f"  {l:10s} P {P:.3f} R {R:.3f} F1 {F1:.3f}")


# -------------------------------------------------------------------------------------
# E8. Predict fold3 with the best weights and package for Codabench
# -------------------------------------------------------------------------------------
best = torch.load("/kaggle/working/c5_lorma_best_trainable.pt")
model.load_state_dict(best, strict=False)
test_ex, test_cache = build_examples(test_text)
test_pred = predict(test_ex, test_cache)
out = {d: [[s, e, l] for s, e, l in test_pred.get(d, [])] for d in test_text}
assert set(out) == set(test_text)
with open("/kaggle/working/prediction.json", "w") as f:
    json.dump(out, f)
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("entities predicted:", sum(len(v) for v in out.values()), "→ /kaggle/working/submission.zip")

LORMA identity-at-init check: max |diff| = 0.00e+00 (should be ~0)
wrapped 32 linears | trainable 1.091M / total 99.50M (1.10%)
train windows 24481 | val windows 24519


Epoch 1/10:   0%|          | 0/383 [00:00<?, ?it/s]

{'epoch': 0, 'train_loss': 0.46888192785860044, 'val_macro_f1': 0.0469513514967618}


Epoch 2/10:   0%|          | 0/383 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.2728277760370593, 'val_macro_f1': 0.06391888278425943}


Epoch 3/10:   0%|          | 0/383 [00:00<?, ?it/s]

... (11 outputs left out)

Epoch 9/10:   0%|          | 0/383 [00:00<?, ?it/s]

{'epoch': 8, 'train_loss': 0.10188955942523853, 'val_macro_f1': 0.09757124646771682}


Epoch 10/10:   0%|          | 0/383 [00:00<?, ?it/s]

{'epoch': 9, 'train_loss': 0.09076972789187034, 'val_macro_f1': 0.09523937772456059}
best val strict macro-F1 0.0976 | time 41.2 min | peak GPU mem 5.03 GB
  APP        P 0.006 R 0.010 F1 0.007
  RESP       P 0.002 R 0.005 F1 0.003
  A.COUNSEL  P 0.004 R 0.008 F1 0.006
  R.COUNSEL  P 0.014 R 0.052 F1 0.021
  JUDGE      P 0.006 R 0.041 F1 0.011
  WIT        P 0.000 R 0.000 F1 0.000
  AUTH       P 0.085 R 0.109 F1 0.095
  COURT      P 0.412 R 0.586 F1 0.484
  STAT       P 0.201 R 0.200 F1 0.201
  PREC       P 0.014 R 0.039 F1 0.021
  DATE       P 0.209 R 0.399 F1 0.274
  CASENO     P 0.013 R 0.042 F1 0.020
entities predicted: 4797 → /kaggle/working/submission.zip


In [7]:
!ls -l /kaggle/working

total 4508
-rw-r--r-- 1 root root 4384797 Oct 11 13:44 c5_lorma_best_trainable.pt
-rw-r--r-- 1 root root  113990 Oct 11 13:50 prediction.json
-rw-r--r-- 1 root root  114118 Oct 11 13:50 submission.zip


In [10]:

# =====================================================================================
# CELL E-KD (c-5) — Knowledge distillation INSIDE the c-5 notebook
#
# TEACHER = fine-tuned c-5 NER model (LORMA + its trained head)
# STUDENT = shallower c-5 model initialized from the Phase 1 checkpoint
#
# Run after Cells 0, A-D and Cell E.
# =====================================================================================

import copy, math, time, json, zipfile, random, os
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm


# -------------------------------------------------------------------------------------
# [H] Hyperparameters — intentionally left unfilled
# -------------------------------------------------------------------------------------
TEACHER_BEST_PATH = "/kaggle/working/c5_lorma_best_trainable.pt"

KEEP_LAYERS    = [0, 2, 4, 7]
TEMPERATURE    = 2.0
KD_WEIGHT      = 0.5
S_LR           = 1e-5
S_EPOCHS       = 10
S_BATCH_SIZE   = 64
S_WEIGHT_DECAY = 0.0
SEED           = 42

for _n in [
    "KEEP_LAYERS", "TEMPERATURE", "KD_WEIGHT",
    "S_LR", "S_EPOCHS", "S_BATCH_SIZE"
]:
    assert globals()[_n] is not None, f"set {_n}"

assert all(0 <= i < cfg.n_layers for i in KEEP_LAYERS) \
       and len(KEEP_LAYERS) < cfg.n_layers, \
       "student must keep fewer blocks than the teacher"

TRAIL_CHARS = globals().get("TRAIL_CHARS", ",.;:")


# -------------------------------------------------------------------------------------
# Span trimming and evaluation
# -------------------------------------------------------------------------------------
def trim(pred, cache):
    """Drop trailing punctuation from predicted spans and discard empty spans."""
    out = {}

    for d, sp in pred.items():
        text, keep = cache[d][0], []

        for s, e, l in sp:
            while TRAIL_CHARS and e > s and text[e - 1] in TRAIL_CHARS:
                e -= 1

            if e > s:
                keep.append((s, e, l))

        out[d] = keep

    return out


def evaluate(m, examples, cache, gold, name):
    """Measure fold2 F1, parameter count, inference time and inference memory."""
    global model
    model = m

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    pred = trim(predict(examples, cache), cache)

    torch.cuda.synchronize()

    res = {
        "name": name,
        "t_inf": time.time() - t0,
        "mem": torch.cuda.max_memory_allocated() / 2**30,
        "params": sum(p.numel() for p in m.parameters())
    }

    res["f1"], res["per"] = strict_macro_f1(pred, gold)

    return res


# -------------------------------------------------------------------------------------
# 1. Teacher: load best weights and evaluate on fold2
# -------------------------------------------------------------------------------------
teacher = model

if os.path.exists(TEACHER_BEST_PATH):
    _miss, _unexp = teacher.load_state_dict(
        torch.load(TEACHER_BEST_PATH, map_location="cpu"),
        strict=False
    )

    assert not _unexp, (
        f"Weights file has keys the teacher lacks: {_unexp[:5]}"
    )

    print(f"Teacher: loaded best weights from {TEACHER_BEST_PATH}")

else:
    print(
        f"WARNING: {TEACHER_BEST_PATH} not found; "
        "using the teacher weights currently in memory"
    )

teacher.to(device).eval()

val_ex, val_cache = build_examples(f2_text, f2_gold)

T_res = evaluate(
    teacher, val_ex, val_cache, f2_gold, "teacher c-5"
)

print(
    f"TEACHER fold2 strict macro-F1 {T_res['f1']:.4f} | "
    f"params {T_res['params']/1e6:.2f}M"
)

if T_res["f1"] < 0.10:
    print(
        "NOTE: weak teacher. The student can only learn what the teacher "
        "knows; consider this when interpreting KD results."
    )


# -------------------------------------------------------------------------------------
# 2. Generate teacher logits for every fold1 token
# -------------------------------------------------------------------------------------
@torch.no_grad()
def teacher_logits(texts):
    ex, cache = build_examples(texts)

    out = {
        d: torch.zeros(
            len(cache[d][1]), len(TAGS), dtype=torch.float16
        )
        for d in cache
    }

    seen = {
        d: torch.zeros(len(cache[d][1]), dtype=torch.bool)
        for d in cache
    }

    for i in tqdm(
        range(0, len(ex), BATCH_SIZE),
        desc="Teacher logits"
    ):
        ids, _, batch = collate(ex[i:i + BATCH_SIZE])

        with torch.autocast("cuda", dtype=torch.float16):
            lg = teacher(ids.to(device)).float().cpu()

        for b, e in enumerate(batch):
            j0, j1 = e["first"], len(e["ids"])

            out[e["doc"]][
                e["start"] + j0:e["start"] + j1
            ] = lg[b, j0:j1].half()

            seen[e["doc"]][
                e["start"] + j0:e["start"] + j1
            ] = True

    assert all(s.all() for s in seen.values()), (
        "Some tokens got no teacher logits"
    )

    assert all(torch.isfinite(v).all() for v in out.values()), (
        "Non-finite teacher logits"
    )

    return out, cache


soft_f1, f1_cache = teacher_logits(f1_text)

_tp = trim(
    {
        d: from_bio(
            f1_cache[d][0],
            f1_cache[d][1],
            [
                TAGS[i]
                for i in soft_f1[d].float().argmax(-1).tolist()
            ]
        )
        for d in soft_f1
    },
    f1_cache
)

print(
    "Teacher fold1 strict macro-F1 from stored logits:",
    round(strict_macro_f1(_tp, f1_gold)[0], 4)
)

del _tp

# Free GPU memory for the student
teacher.cpu()
torch.cuda.empty_cache()


# -------------------------------------------------------------------------------------
# 3. Student: fresh Phase 1 c-5 weights with fewer transformer blocks
# -------------------------------------------------------------------------------------
_ck = torch.load(
    f"{ROOT}/legal-gpt2-final/ckpt.pt",
    map_location="cpu",
    weights_only=False
)

lm_s = DecoderLM(LMConfig(**_ck["config"]))
lm_s.load_state_dict(_ck["model"])

del _ck

# Keep only the selected transformer blocks
lm_s.blocks = nn.ModuleList([
    lm_s.blocks[i] for i in KEEP_LAYERS
])

lm_s.cfg = copy.copy(lm_s.cfg)
lm_s.cfg.n_layers = len(KEEP_LAYERS)


# -------------------------------------------------------------------------------------
# [S] Student trainable parameters
#
# Configuration implemented here:
#   - Freeze token embeddings
#   - Train all retained transformer blocks
#   - Train final normalization
#   - Train the fresh NER head
# -------------------------------------------------------------------------------------
for p in lm_s.parameters():
    p.requires_grad = False

for p in lm_s.blocks.parameters():
    p.requires_grad = True

for p in lm_s.ln_f.parameters():
    p.requires_grad = True

# The fresh NER head in NERModel is trainable by default.
student = NERModel(lm_s, len(TAGS)).to(device)

assert student.head is not None, "NERModel has no head"

n_tr = sum(
    p.numel() for p in student.parameters() if p.requires_grad
)

n_all = sum(
    p.numel() for p in student.parameters()
)

print(
    f"Student: {len(KEEP_LAYERS)} of {cfg.n_layers} blocks "
    f"{list(KEEP_LAYERS)} | "
    f"trainable {n_tr/1e6:.3f}M / "
    f"total {n_all/1e6:.2f}M "
    f"({100*n_tr/n_all:.2f}%)"
)


# -------------------------------------------------------------------------------------
# 4. Training data: fold1 gold labels + teacher logits
# -------------------------------------------------------------------------------------
BATCH_SIZE = S_BATCH_SIZE

train_ex, _ = build_examples(f1_text, f1_gold)

for e in train_ex:
    e["soft"] = soft_f1[e["doc"]][
        e["start"]:e["start"] + len(e["ids"])
    ]

print(
    f"Train windows {len(train_ex)} | "
    f"Val windows {len(val_ex)}"
)


def kd_collate(batch):
    """
    Return:
      ids       : (B, T)
      labels    : (B, T)
      soft      : (B, T, 25)
      soft_mask : (B, T), True where gold labels are valid
    """
    ids, lab, batch = collate(batch)

    soft = torch.zeros(
        *ids.shape, len(TAGS), dtype=torch.float32
    )

    for i, b in enumerate(batch):
        soft[i, :len(b["ids"])] = b["soft"].float()

    return ids, lab, soft, lab != IGNORE


# -------------------------------------------------------------------------------------
# [K] Knowledge distillation loss
#
# L = (1 - KD_WEIGHT) * CE + KD_WEIGHT * KD
#
# CE: gold-label cross-entropy
# KD: temperature-scaled KL divergence between teacher and student
# -------------------------------------------------------------------------------------
def kd_loss(logits, labels, soft, soft_mask):
    student_logits = logits.float()
    teacher_logits = soft.float()
    labels = labels.long()

    # 1. Supervised cross-entropy
    loss_ce = F.cross_entropy(
        student_logits.reshape(-1, student_logits.size(-1)),
        labels.reshape(-1),
        ignore_index=IGNORE
    )

    # 2. Valid positions for distillation
    valid = soft_mask.bool() & (labels != IGNORE)

    if valid.any():
        T = TEMPERATURE

        # Student log-probabilities
        student_log_probs = F.log_softmax(
            student_logits / T, dim=-1
        )

        # Teacher probabilities
        teacher_probs = F.softmax(
            teacher_logits / T, dim=-1
        )

        # Per-token KL divergence, summed across tag classes
        token_kl = F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="none"
        ).sum(dim=-1)

        # Average valid-token KD loss and apply temperature correction
        loss_kd = (T ** 2) * token_kl[valid].mean()

    else:
        # Differentiable zero when no tokens have a valid KD target
        loss_kd = student_logits.sum() * 0.0

    # 3. Combine the losses
    return (
        (1.0 - KD_WEIGHT) * loss_ce
        + KD_WEIGHT * loss_kd
    )


# -------------------------------------------------------------------------------------
# 5. Training loop
# -------------------------------------------------------------------------------------
random.seed(SEED)
torch.manual_seed(SEED)

model = student

params = [
    p for p in student.parameters()
    if p.requires_grad
]

opt = torch.optim.AdamW(
    params,
    lr=S_LR,
    weight_decay=S_WEIGHT_DECAY
)

scaler = torch.amp.GradScaler("cuda")

SAVE = "/kaggle/working/c5_kd_student_best_trainable.pt"

torch.cuda.reset_peak_memory_stats()

t0 = time.time()
best_f1 = -1.0
log = []

for epoch in range(S_EPOCHS):
    student.train()
    random.shuffle(train_ex)

    tot, n, skipped = 0.0, 0, 0

    bar = tqdm(
        range(0, len(train_ex), S_BATCH_SIZE),
        desc=f"Epoch {epoch + 1}/{S_EPOCHS}",
        unit="batch"
    )

    for i in bar:
        ids, lab, soft, smask = kd_collate(
            train_ex[i:i + S_BATCH_SIZE]
        )

        ids = ids.to(device)
        lab = lab.to(device)
        soft = soft.to(device)
        smask = smask.to(device)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = student(ids)

        loss = kd_loss(
            logits.float(),
            lab,
            soft,
            smask
        )

        if not torch.isfinite(loss):
            skipped += 1
            opt.zero_grad(set_to_none=True)
            continue

        opt.zero_grad(set_to_none=True)

        scaler.scale(loss).backward()

        scaler.unscale_(opt)

        torch.nn.utils.clip_grad_norm_(
            params, 1.0
        )

        scaler.step(opt)
        scaler.update()

        tot += loss.item()
        n += 1

        bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{tot / max(n, 1):.4f}",
            skipped=skipped
        )

    model = student

    f1, _ = strict_macro_f1(
        trim(predict(val_ex, val_cache), val_cache),
        f2_gold
    )

    log.append({
        "epoch": epoch,
        "train_loss": tot / max(1, n),
        "val_macro_f1": f1,
        "skipped": skipped
    })

    print(log[-1])

    if f1 > best_f1:
        best_f1 = f1

        # Save trainable parameters only
        torch.save(
            {
                k: p.detach().cpu()
                for k, p in student.named_parameters()
                if p.requires_grad
            },
            SAVE
        )

train_mem = torch.cuda.max_memory_allocated() / 2**30

print(
    f"Student best fold2 F1 {best_f1:.4f} | "
    f"train time {(time.time() - t0)/60:.1f} min | "
    f"peak train GPU mem {train_mem:.2f} GB"
)


# -------------------------------------------------------------------------------------
# 6. Teacher vs student comparison
# -------------------------------------------------------------------------------------
student.load_state_dict(
    torch.load(SAVE, map_location="cpu"),
    strict=False
)

S_res = evaluate(
    student,
    val_ex,
    val_cache,
    f2_gold,
    f"student c-5 x{len(KEEP_LAYERS)} blocks"
)

print(
    f"\n{'model':28s} {'fold2 F1':>9s} "
    f"{'params':>9s} {'infer s':>8s} {'peak GB':>8s}"
)

for r in (T_res, S_res):
    print(
        f"{r['name']:28s} "
        f"{r['f1']:9.4f} "
        f"{r['params']/1e6:8.2f}M "
        f"{r['t_inf']:8.1f} "
        f"{r['mem']:8.2f}"
    )

print(
    f"Student keeps "
    f"{100*S_res['f1']/max(T_res['f1'], 1e-9):.1f}% "
    f"of teacher F1 with "
    f"{100*S_res['params']/T_res['params']:.1f}% "
    f"of its parameters"
)

print("Per-label student scores:")

for l, (P, R, F1) in S_res["per"].items():
    print(
        f"  {l:10s} P {P:.3f} R {R:.3f} F1 {F1:.3f}"
    )


# -------------------------------------------------------------------------------------
# 7. Student predictions on fold3 -> submission.zip
# -------------------------------------------------------------------------------------
test_ex, test_cache = build_examples(test_text)

model = student

test_pred = trim(
    predict(test_ex, test_cache),
    test_cache
)

out = {
    d: [
        [s, e, l]
        for s, e, l in test_pred.get(d, [])
    ]
    for d in test_text
}

assert set(out) == set(test_text)

with open("/kaggle/working/prediction.json", "w") as f:
    json.dump(out, f)

with zipfile.ZipFile(
    "/kaggle/working/submission.zip", "w"
) as z:
    z.write(
        "/kaggle/working/prediction.json",
        arcname="prediction.json"
    )

print(
    "Entities predicted:",
    sum(len(v) for v in out.values()),
    "-> /kaggle/working/submission.zip"
)


Teacher: loaded best weights from /kaggle/working/c5_lorma_best_trainable.pt
TEACHER fold2 strict macro-F1 0.1287 | params 99.50M


Teacher logits:   0%|          | 0/383 [00:00<?, ?it/s]

Teacher fold1 strict macro-F1 from stored logits: 0.338
Student: 4 of 8 blocks [0, 2, 4, 7] | trainable 44.111M / total 54.35M (81.16%)
Train windows 24481 | Val windows 24519


Epoch 1/10:   0%|          | 0/383 [00:00<?, ?batch/s]

{'epoch': 0, 'train_loss': 1.00406463650437, 'val_macro_f1': 0.027968961578772617, 'skipped': 0}


Epoch 2/10:   0%|          | 0/383 [00:00<?, ?batch/s]

... (13 outputs left out)

Epoch 9/10:   0%|          | 0/383 [00:00<?, ?batch/s]

{'epoch': 8, 'train_loss': 0.1176057234017407, 'val_macro_f1': 0.127192708263226, 'skipped': 0}


Epoch 10/10:   0%|          | 0/383 [00:00<?, ?batch/s]

{'epoch': 9, 'train_loss': 0.10611250881815704, 'val_macro_f1': 0.13380111225445843, 'skipped': 0}
Student best fold2 F1 0.1338 | train time 22.7 min | peak train GPU mem 3.60 GB

model                         fold2 F1    params  infer s  peak GB
teacher c-5                     0.1287    99.50M     75.8     0.75
student c-5 x4 blocks           0.1338    54.35M     34.0     1.11
Student keeps 104.0% of teacher F1 with 54.6% of its parameters
Per-label student scores:
  APP        P 0.000 R 0.000 F1 0.000
  RESP       P 0.000 R 0.000 F1 0.000
  A.COUNSEL  P 0.006 R 0.016 F1 0.009
  R.COUNSEL  P 0.008 R 0.010 F1 0.009
  JUDGE      P 0.022 R 0.041 F1 0.028
  WIT        P 0.000 R 0.000 F1 0.000
  AUTH       P 0.125 R 0.085 F1 0.102
  COURT      P 0.619 R 0.744 F1 0.676
  STAT       P 0.231 R 0.252 F1 0.241
  PREC       P 0.021 R 0.052 F1 0.030
  DATE       P 0.392 R 0.627 F1 0.482
  CASENO     P 0.021 R 0.045 F1 0.029
Entities predicted: 6314 -> /kaggle/working/submission.zip
